In [ ]:
!pip install ultralytics
!pip install lime
!pip install opencv-python
!pip install numpy


In [ ]:
from google.colab import drive #mounting google drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [ ]:
from ultralytics import YOLO

# Load the pre-trained YOLOv8 model
model = YOLO("/content/gdrive/MyDrive/Yolo_images/Trained_model/best (1).pt") #Using the model weights from google drive



In [ ]:
import numpy as np
import cv2

def yolo_predict(image):
    # YOLOv8 expects images in the correct format, and preprocessing is handled internally
    results = model(image)

    # Access predictions
    predictions = results[0]

    # Extract the bounding boxes, confidences, and labels
    boxes = predictions.boxes.xyxy.cpu().numpy()  # x1, y1, x2, y2
    confidences = predictions.boxes.conf.cpu().numpy()
    labels = predictions.boxes.cls.cpu().numpy().astype(int)

    return confidences, labels, boxes


In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt

# Initialize LIME image explainer
explainer = lime_image.LimeImageExplainer()

def predict_fn(images):
    predictions = []
    for image in images:
        confidences, labels, boxes = yolo_predict(image)
        # For simplicity, we aggregate confidences for each class
        preds = np.zeros((len(model.names),))
        for conf, label in zip(confidences, labels):
            preds[label] += conf
        predictions.append(preds)
    return np.array(predictions)

# Load an image
image_path = '/content/edb67c20-32_6_1.png'
image = cv2.imread(image_path)

# Ensure the image is correctly loaded
if image is None:
    raise ValueError(f"Image at path {image_path} could not be loaded.")

# Explain the image
explanation = explainer.explain_instance(image, predict_fn, top_labels=3, hide_color=0, num_samples=1000)

# Get the explanation for the top label
temp, mask = explanation.get_image_and_mask(explanation.top_labels[0], positive_only=False, num_features=4, hide_rest=False)
img_boundry1 = mark_boundaries(temp / 255.0, mask)

# Display the image with explanations
plt.imshow(img_boundry1)
plt.axis('off')
plt.show()


In [ ]:
import numpy as np
import cv2
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt

# YOLO prediction function
def yolo_predict(image):
    results = model(image)
    predictions = results[0]
    boxes = predictions.boxes.xyxy.cpu().numpy()  # x1, y1, x2, y2
    confidences = predictions.boxes.conf.cpu().numpy()
    labels = predictions.boxes.cls.cpu().numpy().astype(int)
    return confidences, labels, boxes

# LIME explainer initialization
explainer = lime_image.LimeImageExplainer()

# Prediction function for LIME
def predict_fn(images):
    predictions = []
    for image in images:
        confidences, labels, boxes = yolo_predict(image)
        preds = np.zeros((len(model.names),))
        for conf, label in zip(confidences, labels):
            preds[label] = max(preds[label], conf)
        predictions.append(preds)
    return np.array(predictions)

# Load and preprocess the image
image_path = '/content/fcc24328-29_10.png' # path of the image to be explained
image = cv2.imread(image_path)

if image is None:
    raise ValueError(f"Image at path {image_path} could not be loaded.")

# Explain the image using LIME
explanation = explainer.explain_instance(image, predict_fn, top_labels=3, hide_color=0, num_samples=100)

# Get the explanation for the top label
temp, mask = explanation.get_image_and_mask(explanation.top_labels[0], positive_only=False, num_features=4, hide_rest=False)
img_boundry1 = mark_boundaries(temp / 255.0, mask)

# Display the image with explanations
plt.imshow(img_boundry1)
plt.axis('off')
plt.show()
